# Funstation UK Demand Forecasting — Analysis Notebook

**Nithin Arisetty, 2024**

---

This notebook walks through the methodology for building a regional demand index for Funstation venues across the UK. The goal is to turn publicly available school holiday calendars + BRC footfall data into an operational tool that venue managers and the finance team can actually use for staffing and budget planning.

## 1. Business Context

Funstation is a UK family entertainment centre (FEC) operator with 14+ venues. Looking at their public filings and press coverage, a few things stand out:

- **Revenue is almost entirely holiday-driven.** Their busiest weeks are school holidays, half-terms, and bank holiday weekends. Outside those periods, a significant share of venues likely run near breakeven or below.

- **14 venues ≠ one demand pattern.** Their estate covers Scotland (Edinburgh, Glasgow, Aberdeen), Northern Ireland (Belfast), England across multiple regions, and Wales. Scotland's summer holiday starts in late June. England's starts in late July. That's a 3-4 week gap — if you plan nationally you're either under-staffed in Edinburgh early-summer or over-staffed in Northampton.

- **Staffing, pricing, and marketing all need to be calibrated to local demand peaks.** A social campaign launched in mid-July is too late for Edinburgh but just right for Birmingham.

**My approach:** build a weekly demand index for each region by combining:
1. Is this a school holiday week? (binary, sourced from gov.uk)
2. Is there a bank holiday in this week? (binary, gov.uk JSON API)
3. What's the BRC footfall trend? (continuous modifier, BRC-Sensormatic reports)

Weights:
- School holiday week → **2.0x baseline**
- Bank holiday week → **1.5x baseline**
- Footfall YoY modifier → multiplicative continuous adjustment
- Overlapping factors take the max, not the product (a BH inside summer doesn't double demand, families are already there)

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# add the data directory to path
DATA_DIR = os.path.join('..', 'data')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')
sys.path.insert(0, DATA_DIR)

plt.rcParams.update({
    'figure.facecolor': '#1A1A2E',
    'axes.facecolor': '#16213E',
    'axes.labelcolor': '#EAEAEA',
    'xtick.color': '#EAEAEA',
    'ytick.color': '#EAEAEA',
    'text.color': '#EAEAEA',
    'axes.edgecolor': '#444',
    'grid.color': '#333',
    'grid.alpha': 0.5,
})
ORANGE = '#FF6B35'
TEAL = '#4ECDC4'
print('Setup done')

## 2. Data Loading and Exploration

In [ ]:
# Build from source if processed files don't exist yet
demand_path = os.path.join(PROCESSED_DIR, 'demand_index.csv')
if not os.path.exists(demand_path):
    print('Building demand index from source...')
    from build_demand_index import build_full_demand_index
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    demand_df = build_full_demand_index()
    demand_df.to_csv(demand_path, index=False)
    print('Done')
else:
    demand_df = pd.read_csv(demand_path, parse_dates=['week_start', 'week_end'])

hols_path = os.path.join(PROCESSED_DIR, 'term_dates.csv')
if not os.path.exists(hols_path):
    from fetch_term_dates import build_all_term_dates
    hols_df = build_all_term_dates()
    hols_df.to_csv(hols_path, index=False)
else:
    hols_df = pd.read_csv(hols_path, parse_dates=['start_date', 'end_date'])

footfall_path = os.path.join(PROCESSED_DIR, 'footfall_index.csv')
if not os.path.exists(footfall_path):
    from fetch_footfall import load_footfall_data
    footfall_df = load_footfall_data()
    footfall_df.to_csv(footfall_path, index=False)
else:
    footfall_df = pd.read_csv(footfall_path, parse_dates=['month'])

print(f'Demand index: {demand_df.shape} — {demand_df["region"].nunique()} regions, {demand_df["iso_year"].unique()} years')
print(f'Term dates: {hols_df.shape}')
print(f'Footfall data: {footfall_df.shape}')

In [ ]:
# Quick look at the term dates distribution
print('Holiday periods by region and type:')
print(hols_df.groupby(['region', 'holiday_type']).size().unstack(fill_value=0))
print()
print('Date range in hols_df:')
print(f"  {hols_df['start_date'].min().date()} to {hols_df['end_date'].max().date()}")

In [ ]:
# How long are the different holiday types on average?
hols_df['duration_days'] = (hols_df['end_date'] - hols_df['start_date']).dt.days + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Duration by holiday type
type_duration = hols_df.groupby('holiday_type')['duration_days'].mean().sort_values(ascending=True)
type_duration.plot(kind='barh', ax=axes[0], color=[ORANGE, TEAL, '#FFE66D'])
axes[0].set_title('Avg Duration by Holiday Type (days)', pad=10)
axes[0].set_xlabel('Days')
axes[0].axvline(x=5, color='#888', linestyle='--', alpha=0.5, label='1 week')
axes[0].legend()

# Count by region
region_count = hols_df.groupby('region').size().sort_values(ascending=True)
region_count.plot(kind='barh', ax=axes[1], color=ORANGE)
axes[1].set_title('Holiday Periods per Region', pad=10)
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.suptitle('Term Date Distribution', y=1.02, fontsize=14, color='#EAEAEA')
plt.savefig(os.path.join('..', 'data', 'processed', 'fig_term_date_dist.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Footfall trend — monthly YoY change for shopping centres
# scotland dates are in a different format, needed manual fix
sc_footfall = footfall_df[footfall_df['destination_type'] == 'shopping_centre'].copy()
sc_footfall = sc_footfall.sort_values('month')

fig, ax = plt.subplots(figsize=(14, 4))
colors = [ORANGE if v >= 0 else '#E74C3C' for v in sc_footfall['footfall_yoy_pct']]
bars = ax.bar(sc_footfall['month'], sc_footfall['footfall_yoy_pct'], color=colors, width=20, alpha=0.85)
ax.axhline(y=0, color='#888', linewidth=1)
ax.set_title('BRC-Sensormatic Footfall YoY % Change — Shopping Centres', pad=10)
ax.set_ylabel('YoY %')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join('..', 'data', 'processed', 'fig_footfall_trend.png'), dpi=120, bbox_inches='tight')
plt.show()
print(f'\nAverage footfall modifier: {sc_footfall["footfall_modifier"].mean():.3f}')

### Data Quality Notes

A few things I noticed and handled:

1. **Scotland dates format** — the mygov.scot page uses a different date format than gov.uk. Had to extend the date parser to handle it (`_parse_date_range` in `fetch_term_dates.py`).

2. **Northern Ireland October 2025** — NI's October half-term falls in a different week than England's. Confirmed against nidirect.gov.uk calendar. This is operationally meaningful for the Belfast venue.

3. **BRC data coverage** — BRC doesn't freely publish raw CSVs, only press releases. The figures in the fallback are drawn from their monthly press releases. Should be refreshed when new data is published.

4. **ISO week boundaries** — ISO week 1 of a year sometimes starts in December of the prior year and ISO week 52/53 can straddle year-end. Handled by using `pd.Timestamp.fromisocalendar()` rather than doing the arithmetic manually.

## 3. Demand Index Analysis

In [ ]:
# Top 10 peak weeks per region for 2025
peak_weeks_2025 = (
    demand_df[demand_df['iso_year'] == 2025]
    .sort_values('demand_index', ascending=False)
    .groupby('region')
    .head(10)
    .sort_values(['region', 'demand_index'], ascending=[True, False])
)

print('Top 10 peak weeks by region — 2025')
print(peak_weeks_2025[['region', 'week_start', 'demand_index', 'is_school_holiday', 'is_bank_holiday_week']]
      .to_string(index=False))

In [ ]:
# Compare peak week timing across regions — does Scotland diverge significantly?
top3_per_region = (
    demand_df[demand_df['iso_year'] == 2025]
    .sort_values('demand_index', ascending=False)
    .groupby('region')
    .head(3)
    .groupby('region')['week_number']
    .apply(list)
)

print('Top 3 peak week numbers per region (2025):')
print(top3_per_region.to_string())
print()
print('Edinburgh vs Northampton peak divergence is driven by Scotland summer starting ~3 weeks earlier')

In [ ]:
# 2025 vs 2026 — has Easter shifted?
for year in [2025, 2026]:
    easter_rows = demand_df[
        (demand_df['iso_year'] == year) &
        (demand_df['is_school_holiday'] == True) &
        (demand_df['week_start'].dt.month.isin([3, 4]))
    ].groupby('region')['week_start'].min()
    print(f'First Easter holiday week {year}:')
    print(easter_rows.to_string())
    print()

## 4. Visualisations

In [ ]:
# Heatmap: weeks × regions, colour = demand index
for year in [2025, 2026]:
    df_year = demand_df[demand_df['iso_year'] == year]
    pivot = df_year.pivot_table(index='region', columns='week_number', values='demand_index', aggfunc='mean')

    fig, ax = plt.subplots(figsize=(20, 5))
    im = ax.imshow(
        pivot.values,
        aspect='auto',
        cmap='YlOrRd',
        vmin=0.8,
        vmax=3.0,
        interpolation='nearest'
    )
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xticks(range(0, len(pivot.columns), 4))
    ax.set_xticklabels([f'W{w}' for w in pivot.columns[::4]], rotation=45, ha='right', fontsize=8)
    ax.set_title(f'Weekly Demand Index by Region — {year}', fontsize=13, pad=12)
    plt.colorbar(im, ax=ax, label='Demand Index')
    plt.tight_layout()
    plt.savefig(os.path.join('..', 'data', 'processed', f'fig_heatmap_{year}.png'), dpi=130, bbox_inches='tight')
    plt.show()

In [ ]:
# Rolling 4-week demand index by region — 2025
fig, ax = plt.subplots(figsize=(16, 6))

regions_to_plot = ['Scotland', 'England - Midlands', 'Northern Ireland', 'England - South']
colors_map = [ORANGE, TEAL, '#FFE66D', '#C56BFF']

for region, color in zip(regions_to_plot, colors_map):
    r_df = demand_df[(demand_df['region'] == region) & (demand_df['iso_year'] == 2025)].sort_values('week_start')
    rolling = r_df['demand_index'].rolling(4, min_periods=1).mean()
    ax.plot(r_df['week_number'], rolling, label=region, color=color, linewidth=2)

ax.axhline(y=1.0, color='#555', linestyle=':', linewidth=1, label='Baseline (1.0x)')
ax.axhline(y=1.5, color=ORANGE, linestyle='--', linewidth=1, alpha=0.5, label='Staff trigger (1.5x)')
ax.set_xlabel('ISO Week Number')
ax.set_ylabel('Rolling 4-Week Demand Index')
ax.set_title('Rolling 4-Week Demand Index by Region — 2025', fontsize=13)
ax.legend(loc='upper right', framealpha=0.3)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join('..', 'data', 'processed', 'fig_rolling_demand.png'), dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# Top 10 peak weeks nationally — 2025
national_2025 = (
    demand_df[demand_df['iso_year'] == 2025]
    .groupby(['week_number', 'week_start'])['demand_index']
    .mean()
    .reset_index()
    .nlargest(10, 'demand_index')
    .sort_values('demand_index')
)
national_2025['week_label'] = national_2025['week_start'].dt.strftime('%d %b')

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(national_2025['week_label'], national_2025['demand_index'], color=ORANGE, alpha=0.9)
ax.axvline(x=1.5, color='#888', linestyle='--', label='1.5x threshold')
for bar, val in zip(bars, national_2025['demand_index']):
    ax.text(val + 0.02, bar.get_y() + bar.get_height() / 2, f'{val:.2f}x', va='center', fontsize=9)
ax.set_xlabel('Average Demand Index')
ax.set_title('Top 10 Peak Weeks Nationally — 2025 (Average Across All Regions)', fontsize=12)
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join('..', 'data', 'processed', 'fig_top10_national.png'), dpi=130, bbox_inches='tight')
plt.show()

## 5. Key Findings

### Which weeks drive the highest demand nationally?

Summer (weeks 30-35 in 2025, roughly late July to mid-August) is consistently the highest demand period nationally. But the distribution within that block varies — Scotland peaks in weeks 28-32 while England peaks in weeks 31-35.

Easter is the second-biggest peak, typically a 2-week window in April. The exact dates shift year to year which matters for budget phasing.

October half-term (week 43-44) is the most operationally interesting standalone peak — it's the one high-demand week that doesn't have summer halo effects around it, so venues need to be explicitly staffed up for it.

### Which regions are most holiday-sensitive?

Scotland has the highest coefficient of variation in demand (largest ratio of peak to trough). Long summer holiday + short October break + early February break = more compressed but more extreme peaks.

Northern Ireland has the most distinctive calendar — particularly the July/August period which overlaps with the Twelfth holidays, making demand patterns quite different from England.

England - Midlands (Northampton, Leicester, Birmingham cluster) is broadly representative of the national average, making it a good proxy region for national modelling.

### Practical recommendations for venue ops teams

1. **Stop planning staffing on national averages.** The Streamlit app in this repo lets venue managers filter by their specific region and see their actual peak weeks.

2. **Flag the four standalone peak events explicitly:** Easter, May BH, Summer, October half-term. These need a 4-6 week planning lead time for staffing.

3. **Use the low-demand gap analysis** (SQL file 09) to identify 3-6 week windows for planned maintenance. The longest quiet windows are January and post-October-half-term November.

4. **Review the 2025 vs 2026 calendar shift** — Easter moves by 2 weeks in 2026 which affects Q1/Q2 budget phasing. If Funstation's finance team is allocating marketing spend based on prior-year seasonality, they'll get the Easter push timing wrong in 2026.

---

*TODO: extend this with actual revenue data if it becomes available — the demand index is a proxy, not a revenue model. The correlation with actual visitor numbers would be interesting to validate.*